# Administrative Boundaries (geoBoundaries)

Pulls Indonesia's ADM1 boundaries and filters out Banten (needs `geopandas`, already in the `geoai-suitability` env).

In [ ]:
# make sure we're running from scripts/download so `import common` resolves
import os, sys

NOTEBOOK_DIR = os.path.abspath(".")
if os.path.basename(NOTEBOOK_DIR) != "download":
    NOTEBOOK_DIR = os.path.join(NOTEBOOK_DIR, "scripts", "download")
os.chdir(NOTEBOOK_DIR)
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)
print("cwd:", os.getcwd())

In [ ]:
import requests

from common import RAW_DIR, download_file, ensure_dir, get_logger

log = get_logger("geoboundaries")

API_URL = "https://www.geoboundaries.org/api/current/gbOpen/IDN/ADM1/"
OUT_DIR = RAW_DIR / "boundaries"


def main() -> None:
    ensure_dir(OUT_DIR)

    log.info("Fetching metadata from %s", API_URL)
    resp = requests.get(API_URL, timeout=30)
    resp.raise_for_status()
    meta = resp.json()

    geojson_url = meta["gjDownloadURL"]
    dest = OUT_DIR / "indonesia_adm1.geojson"
    download_file(geojson_url, dest)

    # Split off Banten if geopandas is available
    try:
        import geopandas as gpd

        gdf = gpd.read_file(dest)
        name_col = "shapeName" if "shapeName" in gdf.columns else gdf.columns[0]
        banten = gdf[gdf[name_col].str.contains("Banten", case=False, na=False)]
        if not banten.empty:
            out_path = OUT_DIR / "aoi_banten.geojson"
            banten.to_file(out_path, driver="GeoJSON")
            log.info("Banten AOI saved to %s", out_path)
        else:
            log.warning("Couldn't find 'Banten' under %s, check manually", name_col)
    except ImportError:
        log.info(
            "geopandas isn't installed, skipping the Banten filter. "
            "Open %s in QGIS and filter shapeName == 'Banten' by hand.",
            dest,
        )

In [ ]:
main()